# 📊 Horizon RM Copilot - Quantitative Evaluation Notebook

This notebook runs automated evaluation pipelines against `eval_dataset.json` to calculate key metrics for the wealth management RM Copilot, covering:
1. **Retrieval Coverage & Exact Match Accuracy**
2. **Suitability Gate Precision, Recall & F1-Score**
3. **Ragas Faithfulness & Answer Relevancy (LLM-as-a-Judge)**
4. **Single-Shot vs Multi-Hop Vector Retrieval Analysis**

### Prerequisites
Ensure you have a populated ChromaDB by running `python ingest.py` and have a valid `OPENAI_API_KEY` set in your `.env` file.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Load env variables (contains OpenAI API key)
load_dotenv()

# Verify dataset exists
dataset_path = "eval_dataset.json"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

with open(dataset_path, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print(f"Successfully loaded evaluation dataset: '{eval_data['_meta']['title']}' v{eval_data['_meta']['version']}")
print(f"Categories present: {list(eval_data.keys())[1:]}")

## 🔍 Section 1: Retrieval Relevance & Entitlement Coverage

Here, we evaluate the retriever's ability to fetch the expected documents while strictly filtering out restricted research (such as `RN-002`) for Tier-1 RMs.

In [ ]:
from tools import rag_retriever

retrieval_cases = eval_data["retrieval"]["test_cases"]
retrieval_results = []
successful_hits = 0

for tc in retrieval_cases:
    query = tc["query"]
    rm_tier = tc["rm_tier"]
    expected = tc["expected_doc_ids"]
    must_not = tc["must_not_contain"]
    
    retrieved = rag_retriever(query, rm_research_tier=rm_tier)
    retrieved_ids = [doc["doc_id"] for doc in retrieved]
    
    hits = [exp for exp in expected if exp in retrieved_ids]
    
    if len(expected) == 0:
        is_success = all(mn not in retrieved_ids for mn in must_not)
        coverage = 1.0 if is_success else 0.0
    else:
        coverage = len(hits) / len(expected)
        is_success = (coverage == 1.0) and all(mn not in retrieved_ids for mn in must_not)
        
    if is_success:
        successful_hits += 1
        
    retrieval_results.append({
        "ID": tc["id"],
        "Category": tc["category"],
        "RM Tier": rm_tier,
        "Expected Docs": expected,
        "Retrieved Docs": list(set(retrieved_ids)),
        "Coverage": coverage,
        "Pass": is_success
    })

df_retrieval = pd.DataFrame(retrieval_results)
mrr_score = successful_hits / len(retrieval_cases)
print(f"=== Retrieval Evaluation Complete ===")
print(f"Overall Exact Match Accuracy: {mrr_score * 100:.2f}%")
df_retrieval[["ID", "Category", "RM Tier", "Expected Docs", "Retrieved Docs", "Coverage", "Pass"]]

## 🛡️ Section 2: Suitability Gate Precision & Recall

This evaluates the programmatic checks (`suitability_checker`) against 10 test cases containing risk profile mismatches, concentration breaches, and unknown products. 
We calculate:
* **Precision**: The fraction of flagged transactions that were actually violations.
* **Recall**: The fraction of actual violations that were successfully flagged.
* **F1-Score**: The harmonic mean of precision and recall.

In [ ]:
from tools import suitability_checker

suitability_cases = eval_data["suitability_gate"]["test_cases"]
suitability_results = []

tp = fp = fn = tn = 0

for tc in suitability_cases:
    client_id = tc["client_id"]
    product_code = tc["product_code"]
    amount = tc["allocation_amount"]
    expected_status = tc["expected_status"]
    
    res = suitability_checker(client_id, product_code, amount)
    actual_status = res["status"]
    passed = actual_status == expected_status
    
    # Classify as TP, FP, FN, TN (Positive class = Flagged/Blocked/Needs Review)
    if expected_status in ["Blocked", "Needs Review"]:
        if actual_status in ["Blocked", "Needs Review"]:
            tp += 1
            classification = "True Positive"
        else:
            fn += 1
            classification = "False Negative"
    else: # Expected Cleared
        if actual_status == "Cleared":
            tn += 1
            classification = "True Negative"
        else:
            fp += 1
            classification = "False Positive"
            
    suitability_results.append({
        "ID": tc["id"],
        "Client": client_id,
        "Product": product_code,
        "Expected Status": expected_status,
        "Actual Status": actual_status,
        "Classification": classification,
        "Violations": res.get("violations", []),
        "Pass": passed
    })

df_suitability = pd.DataFrame(suitability_results)

precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 1.0

print("=== Suitability Confusion Matrix Summary ===")
print(f"True Positives (Violations Correctly Blocked): {tp}")
print(f"False Positives (Cleared Incorrectly Blocked): {fp}")
print(f"False Negatives (Violations Missed): {fn}")
print(f"True Negatives (Cleared Correctly Allowed): {tn}")
print("-" * 40)
print(f"Gate Precision: {precision * 100:.2f}%")
print(f"Gate Recall (Sensitivity): {recall * 100:.2f}%")
print(f"Gate F1 Score: {f1 * 100:.2f}%")
df_suitability[["ID", "Client", "Product", "Expected Status", "Actual Status", "Classification", "Pass"]]

## 🧪 Section 3: Ragas LLM-as-a-Judge Pipeline Metrics

This section evaluates the quality of generated briefs in the `faithfulness` subset. 
It measures:
1. **Faithfulness / Groundedness**: Whether all statements in the brief are directly supported by the retrieved context.
2. **Answer Relevancy**: Whether the generated brief directly addresses the user request.
3. **Tool Call Accuracy**: Coverage of expected files.

In [ ]:
import sys
import types

# Stub out langchain_community.chat_models.vertexai to prevent Ragas import errors if not configured
if "langchain_community.chat_models.vertexai" not in sys.modules:
    m = types.ModuleType("vertexai")
    m.ChatVertexAI = None
    sys.modules["langchain_community.chat_models.vertexai"] = m

from client_db import get_client_profile
from agent import create_agent
from analytics import clean_answer_for_eval, format_brief_to_text

faithfulness_cases = eval_data["faithfulness"]["test_cases"]
agent_responses = []

print("Running agent pipeline over test cases...")
for idx, tc in enumerate(faithfulness_cases):
    query = tc["query"]
    client_id = tc["client_id"]
    expected_docs = tc.get("expected_source_doc_ids", [])
    
    profile = get_client_profile(client_id)
    initial_state = {
        "query": query,
        "client_id": client_id,
        "product_code": tc.get("product_code"),
        "allocation_amount": 0.0,
        "force_structured": False,
        "response_mode": "freeform",
        "client_profile": profile,
        "retrieved_evidence": [],
        "compliance_status": "Cleared",
        "escalated": False,
        "review_notes": None,
        "final_brief": None,
        "free_form_response": None,
        "is_out_of_context": None,
        "plan": None,
        "suitability_report": None,
        "draft_brief": None
    }
    
    agent = create_agent(interrupt_before_nodes=[])
    config = {"configurable": {"thread_id": f"notebook_eval_{client_id}_{idx}"}}
    final_state = agent.invoke(initial_state, config)
    
    retrieved_evidence = final_state.get("retrieved_evidence", []) or []
    contexts = [d["content"] for d in retrieved_evidence]
    retrieved_ids = list(set([d["doc_id"] for d in retrieved_evidence if d.get("doc_id")]))
    
    # Calculate tool accuracy
    if expected_docs:
        hits = [exp for exp in expected_docs if exp in retrieved_ids]
        tool_acc = len(hits) / len(expected_docs)
    else:
        tool_acc = 1.0 if not retrieved_ids else 0.0
        
    # Extract response
    if final_state.get("free_form_response"):
        answer = clean_answer_for_eval(final_state["free_form_response"])
    elif final_state.get("final_brief"):
        answer = clean_answer_for_eval(format_brief_to_text(final_state["final_brief"], query))
    else:
        answer = "No response generated."
        
    agent_responses.append({
        "question": query,
        "contexts": contexts,
        "answer": answer,
        "tool_accuracy": tool_acc,
        "retrieved_ids": retrieved_ids,
        "expected_ids": expected_docs
    })
    print(f"[{idx+1}/{len(faithfulness_cases)}] Query processed. Tool Call Accuracy: {tool_acc * 100:.1f}%")

df_agent = pd.DataFrame(agent_responses)
df_agent[["question", "expected_ids", "retrieved_ids", "tool_accuracy"]]

### Ragas Metrics Evaluation (Faithfulness & Relevancy)

Below, we run Ragas scoring. Ensure you have `ragas` and `datasets` installed. 
*(Note: If you do not have Ragas installed, you can run `pip install ragas datasets` before executing this cell).*

In [ ]:
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics._faithfulness import Faithfulness
    from ragas.metrics._answer_relevance import ResponseRelevancy
    from ragas.llms import llm_factory
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import OpenAIEmbeddings
    from openai import OpenAI as OpenAIClient

    openai_api_key = os.environ.get("OPENAI_API_KEY")
    if not openai_api_key:
        print("Warning: OPENAI_API_KEY environment variable not set. Ragas evaluation will be skipped.")
    else:
        # Prepare dataset for Ragas
        ragas_ds = Dataset.from_dict({
            "question": df_agent["question"].tolist(),
            "contexts": df_agent["contexts"].tolist(),
            "answer": df_agent["answer"].tolist()
        })
        
        openai_client = OpenAIClient(api_key=openai_api_key)
        ragas_llm = llm_factory("gpt-4o-mini", client=openai_client)
        lc_emb = OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_api_key)
        ragas_emb = LangchainEmbeddingsWrapper(lc_emb)
        
        faith_metric = Faithfulness()
        faith_metric.llm = ragas_llm
        
        relevancy_metric = ResponseRelevancy()
        relevancy_metric.llm = ragas_llm
        relevancy_metric.embeddings = ragas_emb
        
        print("Running Ragas evaluation scoring...")
        results = evaluate(
            dataset=ragas_ds,
            metrics=[faith_metric, relevancy_metric]
        )
        
        df_agent["Ragas Faithfulness"] = list(results["faithfulness"])
        df_agent["Ragas Relevancy"] = list(results["answer_relevancy"])
        
        print("\n=== Average Ragas Quality Metrics ===")
        print(f"Ragas Faithfulness: {np.nanmean(df_agent['Ragas Faithfulness']) * 100:.2f}%")
        print(f"Ragas Answer Relevancy: {np.nanmean(df_agent['Ragas Relevancy']) * 100:.2f}%")
        print(f"Average Tool Call Accuracy: {df_agent['tool_accuracy'].mean() * 100:.2f}%")
except ImportError as e:
    print(f"Ragas or datasets packages not installed. Run 'pip install ragas datasets' to compute LLM-as-a-judge metrics. Error: {e}")
except Exception as e:
    print(f"An error occurred during Ragas execution: {e}")

## 🔗 Section 4: Single-Shot vs Multi-Hop Retrieval Comparison

Here we compare how a simple, single-shot similarity lookup performs against a multi-hop traversal lookup on cross-referenced research reports.

In [ ]:
from tools import market_data_tool

# Run single shot
single_query = "Summarize the house views on fixed income portfolio duration relative to the Q2 2026 Global Market Outlook."
single_retrieved = rag_retriever(single_query, rm_research_tier=2)
single_ids = list(set([doc["doc_id"] for doc in single_retrieved]))

# Run multi-hop simulation
step1_retrieved = rag_retriever(single_query, rm_research_tier=2)
step1_ids = list(set([doc["doc_id"] for doc in step1_retrieved]))

# Simulate link-following
from agent import get_llm
llm = get_llm()
context_text = "\n\n".join([d["content"][:500] for d in step1_retrieved])

prompt = f"""
Based on the retrieved context below, identify any other document IDs (e.g. RN-003, PG-001, etc.) referenced or linked that are relevant to 'fixed income duration views':

Context:
{context_text}

Return a comma-separated list of document IDs.
"""

referenced_ids = [x.strip() for x in llm.invoke(prompt).content.split(",") if x.strip()]
multi_hop_ids = list(set(step1_ids + referenced_ids))
multi_hop_ids = [i for i in multi_hop_ids if i in ["PG-001", "PG-002", "PG-003", "PG-004", "CMP-001", "CMP-002", "CMP-003", "RN-001", "RN-002", "RN-003"]]

print("=== Retrieval Method Comparison ===")
print(f"Single-Shot Retrieval Doc IDs: {single_ids}")
print(f"Multi-Hop (Blended Links) Doc IDs: {multi_hop_ids}")
print("-" * 50)
if "RN-003" in multi_hop_ids and "RN-003" not in single_ids:
    print("Result: Multi-hop retrieval successfully followed links to RN-003 (Fixed Income Outlook), which was missed in single-shot search.")
else:
    print("Result: Both methods identified critical documents, but multi-hop guarantees linkage tracing.")

## 📊 Section 5: Consolidated Quality Dashboard

Let's assemble a summary of all metrics in a clean visual layout.

In [ ]:
metrics_summary = {
    "Metric Name": [
        "Retriever Exact Match Accuracy",
        "Suitability Gate Precision",
        "Suitability Gate Recall",
        "Suitability Gate F1 Score",
        "Average Tool Call Accuracy"
    ],
    "Score (%)": [
        mrr_score * 100,
        precision * 100,
        recall * 100,
        f1 * 100,
        df_agent["tool_accuracy"].mean() * 100
    ]
}

df_metrics = pd.DataFrame(metrics_summary)
print(df_metrics.to_string(index=False))

# Plotting metrics
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    bars = plt.barh(df_metrics["Metric Name"], df_metrics["Score (%)"], color=['#2b5c8f', '#4682b4', '#5f9ea0', '#20b2aa', '#3cb371'])
    plt.xlim(0, 115)
    plt.xlabel("Percentage (%)")
    plt.title("Horizon RM Copilot Performance Benchmark Dashboard")
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, bar.get_y() + bar.get_height()/2, f"{width:.1f}%", va='center', ha='left', fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\nMatplotlib not installed. Performance numbers printed above.")